Fake vs Real (BoW Baseline)

In [40]:
import os, re
import numpy as np
import pandas as pd
import optuna

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.metrics import f1_score

pd.set_option("display.max_colwidth", None)  # don't truncate strings in DataFrame displays

hyperparameters 


In [41]:
TEST_SIZE        = 0.20
RANDOM_STATE     = 42

# TF-IDF
NGRAM_RANGE      = (1, 2)
MAX_FEATURES     = 50_000
MIN_DF           = 2
STOP_WORDS       = "english"   
# Model
MAX_ITER         = 2000
CLASS_WEIGHT     = "balanced"

# Demo / printing
THRESHOLD        = 0.60      # threshold for "definitely true"
TOPN             = 3           # top contributing n-grams to show
PREVIEW_CHARS    = 800
N_SAMPLES_EACH   = 2           # how many to print per class
MIN_CHARS_REAL   = 700
MIN_CHARS_FAKE   = 700         # lower to 400 if fake articles are shorter

Load & prepare the dataframe
The pick_text function standardizes raw CSV inputs by extracting or combining the most relevant text columns, cleaning them, and ensuring consistency across sources. Afterward, Fake and True datasets are merged into a single dataframe with unified text, label, and source columns. This step creates a clean, reliable foundation for building machine learning models to detect fake versus real news.

In [42]:
def pick_text(df: pd.DataFrame) -> pd.Series:
    """Select a text field (title+body if available, else first string column)."""
    title_cols = ["title", "headline", "subject"]
    body_cols  = ["text", "content", "article", "body"]

    title = next((c for c in title_cols if c in df.columns), None)
    body  = next((c for c in body_cols  if c in df.columns), None)

    if title and body:
        s = df[title].fillna("").astype(str).str.strip() + " " + df[body].fillna("").astype(str).str.strip()
    elif body:
        s = df[body].fillna("").astype(str).str.strip()
    else:
        str_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
        if not str_cols:
            raise ValueError("No text-like columns found.")
        s = df[str_cols[0]].fillna("").astype(str).str.strip()

    s = s.replace(r"\s+", " ", regex=True).str.strip()
    s.name = "text"
    return s

# Load Kaggle Fake/True CSVs from current folder
fake = pd.read_csv("Fake.csv")
true = pd.read_csv("True.csv")

fake["label"]  = 0
true["label"]  = 1
fake["source"] = "Fake.csv"
true["source"] = "True.csv"

df = pd.concat(
    [
        pd.DataFrame({"text": pick_text(fake), "label": 0, "source": fake["source"]}),
        pd.DataFrame({"text": pick_text(true), "label": 1, "source": true["source"]}),
    ],
    ignore_index=True
)


This cell cleans the combined dataset by removing empty rows and duplicate text-label pairs, then resets the index for consistency. It also adds a simple id column to uniquely identify each article. Finally, it prints dataset statistics, showing the total rows and the balance between real and fake news.

In [43]:
# Basic cleanup & trace id
df = df[df["text"].str.strip().ne("")].drop_duplicates(subset=["text", "label"]).reset_index(drop=True)
df["id"] = df.index

print(f"Dataset ready: {len(df)} rows  |  Real={sum(df.label==1)}  Fake={sum(df.label==0)}")

Dataset ready: 39100 rows  |  Real=21195  Fake=17905


splits the dataset into training and testing sets while keeping the real/fake ratio balanced. It then converts the raw text into TF-IDF features, capturing important words and n-grams while filtering noise. Finally, a Logistic Regression classifier is trained on the Bag-of-Words representation to learn patterns that distinguish fake from real news.

In [44]:

# Train/test split & BoW training

X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"],
    test_size=TEST_SIZE, stratify=df["label"], random_state=RANDOM_STATE
)

# Use optuna for hyperparameter optimization

def objective(trial):
    max_features = trial.suggest_int("max_features", 5000, 100_000, step=5000)
    ngram_min = trial.suggest_int("ngram_min", 1, 2)
    ngram_max = trial.suggest_int("ngram_max", ngram_min, 3)
    C = trial.suggest_loguniform("C", 1e-3, 10)

    vectorizer = TfidfVectorizer(
        stop_words = STOP_WORDS,
        max_features = max_features,
        ngram_range = (ngram_min, ngram_max),
        min_df = MIN_DF
    )
    Xtr = vectorizer.fit_transform(X_train)
    Xval = vectorizer.transform(X_test)

    clf = LogisticRegression(
        max_iter = MAX_ITER,
        class_weight = CLASS_WEIGHT,
        C = C,
        solver = "liblinear"
    )
    clf.fit(Xtr, y_train)
    y_pred = clf.predict(Xval)
    return f1_score(y_test, y_pred)

study = optuna.create_study(direction = "maximize")
study.optimize(objective, n_trials=50)

best_params = study.best_trial.params
print("Best hyperparameters: ", best_params)

#Train model

tfidf = TfidfVectorizer(
    stop_words=STOP_WORDS,
    ngram_range=(best_params["ngram_min"], best_params["ngram_max"]),
    max_features=best_params["max_features"],
    min_df=MIN_DF
)
Xtr_final = tfidf.fit_transform(X_train)
Xte_final = tfidf.transform(X_test)

bow_clf = LogisticRegression(max_iter=MAX_ITER, class_weight=CLASS_WEIGHT, C = best_params["C"], solver = "liblinear")
bow_clf.fit(Xtr_final, y_train)

[I 2025-09-27 13:19:22,555] A new study created in memory with name: no-name-2c289bf5-7f1a-494d-8f5c-8d1c458cc4bb
C:\Users\DSU\AppData\Local\Temp\ipykernel_19508\1762942200.py:14: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-3, 10)
[I 2025-09-27 13:19:48,759] Trial 0 finished with value: 0.9888327259903609 and parameters: {'max_features': 100000, 'ngram_min': 2, 'ngram_max': 2, 'C': 5.876035458640878}. Best is trial 0 with value: 0.9888327259903609.
C:\Users\DSU\AppData\Local\Temp\ipykernel_19508\1762942200.py:14: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-3, 10)
[I 2025-09-27 13:20:48,767] Tr

Best hyperparameters:  {'max_features': 5000, 'ngram_min': 1, 'ngram_max': 2, 'C': 6.591533643450495}


,penalty,'l2'
,dual,False
,tol,0.0001
,C,6.591533643450495
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'liblinear'
,max_iter,2000
,multi_class,'deprecated'


This evaluation cell measures how well the BoW model performs on unseen test data. It predicts both class labels and probabilities, then prints precision, recall, and F1-scores for fake vs. real news. The ROC-AUC score provides an overall measure of the model’s ability to distinguish between the two classes.

In [45]:

#  Eval (test split)
y_pred  = bow_clf.predict(Xte_final)
y_proba = bow_clf.predict_proba(Xte_final)[:, 1]

print("\n=== BoW (TF-IDF + Logistic Regression) ===")
print(classification_report(y_test, y_pred, digits=3))
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 4))


=== BoW (TF-IDF + Logistic Regression) ===
              precision    recall  f1-score   support

           0      0.993     0.991     0.992      3581
           1      0.992     0.994     0.993      4239

    accuracy                          0.993      7820
   macro avg      0.993     0.992     0.993      7820
weighted avg      0.993     0.993     0.993      7820

ROC-AUC: 0.9993


This cell defines a compact toolkit for demoing predictions and explanations. predict_news turns raw text into a REAL/FAKE decision with a tunable threshold, while explain_instance surfaces the top n-grams that most influenced that decision (by weight×tf-idf). sample_from_test fetches random, sufficiently long items from the held-out set, and demo_row prints a neat preview plus the top contributors use compact=True for a short, presentation-ready summary.

In [46]:
# Demo functions
def predict_news(text: str, model, vectorizer, thr: float = 0.5):
    """Return ('REAL'|'FAKE', probability_of_REAL)."""
    V = vectorizer.transform([text])
    p = model.predict_proba(V)[0, 1]
    return ("REAL" if p >= thr else "FAKE"), float(p)

def explain_instance(text: str, vectorizer, model, topn: int = 10):
    """Return list of (feature, contribution) sorted by absolute contribution."""
    X1 = vectorizer.transform([text])
    feat = np.array(vectorizer.get_feature_names_out())
    coef = model.coef_[0]
    nz_idx = X1.nonzero()[1]
    nz_vals = X1.data
    contrib = coef[nz_idx] * nz_vals  # weight * tfidf
    order = np.argsort(np.abs(contrib))[::-1][:topn]
    return [(feat[nz_idx[i]], float(contrib[i])) for i in order]

def sample_from_test(df_test: pd.DataFrame, label: int, n: int = 2, min_chars: int = 700, random_state=None):
    """Sample n rows from the test set by label with a minimum length."""
    pool = df_test[(df_test["label"] == label) & (df_test["text"].astype(str).str.len() >= min_chars)]
    if pool.empty:
        raise ValueError(f"No test articles with label={label} and len >= {min_chars}.")
    return pool.sample(n=min(n, len(pool)), random_state=random_state).reset_index(drop=True)

def demo_row(text: str, true_label: int, model, vectorizer,
             threshold: float = THRESHOLD, topn: int = TOPN, preview_chars: int = PREVIEW_CHARS, compact: bool = False):
    """Pretty print a single article’s prediction + top contributors."""
    txt = ("" if text is None else str(text)).strip()
    pred, p = predict_news(txt, model, vectorizer, thr=threshold)
    truth = "REAL" if true_label == 1 else "FAKE"

    print("="*70)
    if compact:
        feats = explain_instance(txt, vectorizer, model, topn=topn)
        tops = ", ".join([f for f, _ in feats])
        print(f"Pred={pred} | True={truth} | p_real={p:.3f} | len={len(txt)}")
        print("-"*70)
        print(txt[:preview_chars] + (" ..." if len(txt) > preview_chars else ""))
        print("Top:", tops)
        return

    print(f"True: {truth} | Predicted: {pred} | p_real={p:.3f}")
    print("-"*70)
    print("Article:\n" + (txt if len(txt) <= preview_chars else txt[:preview_chars] + " ..."))
    print("\nTop contributors:")
    for f, val in explain_instance(txt, vectorizer, model, topn=topn):
        sign = "REAL+" if val > 0 else "FAKE-"
        print(f"  {f:20s} ({sign})")

This cell finalizes the pipeline by pulling long examples from the test set and running them through the trained BoW model. First, it rebuilds df_test with text, labels, and lengths. Then, it randomly samples a few REAL and FAKE articles that meet a minimum length requirement to ensure meaningful context. Finally, it prints model predictions alongside article previews and the top TF-IDF contributors, giving you both accuracy insights and interpretability

In [47]:

#  Build df_test once (global)

df_test = pd.DataFrame({"text": X_test.reset_index(drop=True), "label": y_test.reset_index(drop=True)})
df_test["len"] = df_test["text"].astype(str).str.len()


# random samples from the test split
# REAL samples
real_long = sample_from_test(df_test, label=1, n=N_SAMPLES_EACH, min_chars=MIN_CHARS_REAL, random_state=None)
# FAKE samples
fake_long = sample_from_test(df_test, label=0, n=N_SAMPLES_EACH, min_chars=MIN_CHARS_FAKE, random_state=None)

print("\n########### REAL (test) ###########")
for _, r in real_long.iterrows():
    demo_row(r["text"], true_label=1, model=bow_clf, vectorizer=tfidf,
             threshold=THRESHOLD, topn=TOPN, preview_chars=PREVIEW_CHARS, compact=False)

print("\n########### FAKE (test) ###########")
for _, r in fake_long.iterrows():
    demo_row(r["text"], true_label=0, model=bow_clf, vectorizer=tfidf,
             threshold=THRESHOLD, topn=TOPN, preview_chars=PREVIEW_CHARS, compact=False)






########### REAL (test) ###########
True: REAL | Predicted: REAL | p_real=0.971
----------------------------------------------------------------------
Article:
Trump hasn't sued a newspaper for libel in decades, records show (In this Oct. 13 story, corrects description of legal standard regarding Trump in paragraph 15) By Alison Frankel and Dan Levine (Reuters) - Donald Trump hasn’t sued a newspaper for libel in three decades, despite the Republican presidential nominee repeatedly threatening to do so over the course of his business career, according to databases of state and federal court records. A lawyer for the New York real estate developer demanded on Wednesday The New York Times retract a story in which two women accused Trump of inappropriately touching them. If the newspaper did not comply, Trump, who says the allegations are fabricated, would “pursue all available actions and remedies,” the lawyer, Marc Kasowitz, said in a letter. Trump ...

Top contributors:
  reuters      

This helper function prints a quick, compact summary of model predictions for multiple articles. For each row

In [48]:
#   one-line summary

def print_one_liners(df_in: pd.DataFrame, threshold: float = THRESHOLD, topn: int = TOPN, preview: int = 300):
    """Compact per-article summary: prediction + short preview + top features."""
    for i, row in df_in.iterrows():
        txt = ("" if row["text"] is None else str(row["text"])).strip()
        pred, p = predict_news(txt, bow_clf, tfidf, thr=threshold)
        feats = ", ".join([f for f, _ in explain_instance(txt, tfidf, bow_clf, topn=topn)])
        truth = "REAL" if row["label"] == 1 else "FAKE"
        snippet = txt if len(txt) <= preview else txt[:preview] + " ..."
        print(f"[{i}] Pred={pred} | True={truth} | p_real={p:.3f} | top: {feats}\n    {snippet}\n")

In [49]:
print_one_liners(pd.concat([real_long, fake_long], ignore_index=True), preview=2000)


[0] Pred=REAL | True=REAL | p_real=0.971 | top: reuters, said, story
    Trump hasn't sued a newspaper for libel in decades, records show (In this Oct. 13 story, corrects description of legal standard regarding Trump in paragraph 15) By Alison Frankel and Dan Levine (Reuters) - Donald Trump hasn’t sued a newspaper for libel in three decades, despite the Republican presidential nominee repeatedly threatening to do so over the course of his business career, according to databases of state and federal court records. A lawyer for the New York real estate developer demanded on Wednesday The New York Times retract a story in which two women accused Trump of inappropriately touching them. If the newspaper did not comply, Trump, who says the allegations are fabricated, would “pursue all available actions and remedies,” the lawyer, Marc Kasowitz, said in a letter. Trump said at a rally on Thursday he was preparing a lawsuit. An attorney for the Times, David McCraw, said the story was of nationa

In [50]:
from collections import Counter

def compare_top_features(df, model, vectorizer, label, n_samples=50, topn=5):
    """Aggregate most common influential features for a label subset."""
    sub = df[df["label"]==label].sample(n=n_samples, random_state=42)
    feats = []
    for _, row in sub.iterrows():
        txt = str(row["text"])
        feats.extend([f for f,_ in explain_instance(txt, vectorizer, model, topn=topn)])
    return Counter(feats).most_common(15)

print("REAL top features:", compare_top_features(df_test, bow_clf, tfidf, label=1))
print("FAKE top features:", compare_top_features(df_test, bow_clf, tfidf, label=0))


REAL top features: [('reuters', 50), ('said', 47), ('washington reuters', 11), ('president donald', 8), ('brexit', 5), ('thursday', 5), ('election', 4), ('britain', 4), ('wednesday', 4), ('london reuters', 4), ('ireland', 3), ('london', 3), ('video', 3), ('tuesday', 3), ('saudi', 2)]
FAKE top features: [('said', 21), ('just', 16), ('video', 13), ('read', 11), ('obama', 9), ('gop', 9), ('republican', 6), ('hillary', 6), ('featured image', 6), ('president trump', 6), ('watch', 5), ('com', 4), ('rep', 4), ('america', 4), ('century wire', 3)]


In [51]:
import joblib

# Save
joblib.dump(bow_clf, "bow_logreg_model.pkl")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")

# Load later
bow_clf = joblib.load("bow_logreg_model.pkl")
tfidf   = joblib.load("tfidf_vectorizer.pkl")
